In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

In [3]:
data_file = "titanic_dataset.csv" if Path("titanic_dataset.csv").exists() else "../titanic_dataset.csv"
df = pd.read_csv(data_file)
print("Shape: ", df.shape)

Shape:  (891, 15)


In [4]:
df = df.drop(columns=["alive", "deck"])
target = "survived"
X = df.drop(columns=[target])
y = df[target]
 
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
 
age_median = X_train["age"].median()
fare_median = X_train["fare"].median()
for frame in (X_train, X_test):
    frame["age"] = frame["age"].fillna(age_median)
    frame["fare"] = frame["fare"].fillna(fare_median)
    frame["gender_male"] = (frame["gender"] == "male").astype(int)
 
cols = ["pclass", "age", "fare", "gender_male"]
model = LogisticRegression(max_iter=1000)
model.fit(X_train[cols], y_train)
pred = pd.Series(model.predict(X_test[cols]), index=X_test.index)
print("Overall accuracy:", round(accuracy_score(y_test, pred), 3))

Overall accuracy: 0.788


In [7]:
for gender in ["female","male"]:
    mask = X_test["gender"] == gender
    print(gender)
    print("  rows:",int(mask.sum()))
    print("  actual survival rate:",round(y_test[mask].mean(), 3))
    print("  Predicated survival rate:",round(pred[mask].mean(), 3))
    print("  recall:",round(recall_score(y_test[mask],pred[mask]), 3))

female
  rows: 61
  actual survival rate: 0.738
  Predicated survival rate: 0.951
  recall: 0.956
male
  rows: 118
  actual survival rate: 0.203
  Predicated survival rate: 0.042
  recall: 0.167
